# Momompurin Live Manager — Algorithm Benchmarking

**ALGOCOM PBL Milestone 1 · Group 8 · III-BCSAD · University of Makati**

Prof. Nomar B. Maestro · 7 November 2026

---

This notebook measures the two algorithmic engines the application actually runs, plus the two
supporting structures the workflow depends on. Every benchmark runs against the **client's real
catalogue** — 767 rows exported from the Momompurin Live Manager workbook — not a sample invented
for a slide.

Two things make these results defensible rather than decorative:

1. **Each implementation is paired with the naive baseline** a developer writes first, and the
   notebook **asserts the two produce identical answers**. A fast wrong answer is not a result.
2. **Timings are taken as the best of several repeats**, and the query set is generated from real
   catalogue names with a fixed seed, so a re-run is comparable.

> **Note on absolute numbers.** Wall-clock times depend on the machine. Run this in Colab and the
> milliseconds will differ from the report, which ran on the group's own hardware. The **ratios**
> and the **accuracy figures** are what the argument rests on, and those are stable across
> machines. The report's figures are quoted beside each live result so the two can be compared.


## 0 · Setup

Loads the catalogue straight from the project's public repository. No upload step, no local file.


In [ ]:
import csv
import heapq
import json
import random
import re
import statistics
import time
import urllib.request
from collections import defaultdict

import pandas as pd

CATALOGUE_URL = (
    "https://raw.githubusercontent.com/WhiteQilin/"
    "momompurin-live-manager/main/algorithms/benchmark/catalogue.csv"
)

SEED = 20261104
REPEATS = 5

# ---------------------------------------------------------------- load
with urllib.request.urlopen(CATALOGUE_URL) as r:
    rows = list(csv.DictReader(r.read().decode("utf-8").splitlines()))

SKUS = [row["sku"] for row in rows]
NAMES = [row["item_name"] for row in rows]

print(f"catalogue rows    : {len(rows)}")
print(f"distinct SKUs     : {len(set(SKUS))}")
print()
print("first five entries:")
for sku, name in list(zip(SKUS, NAMES))[:5]:
    print(f"  {sku:<10} {name}")

## 1 · Shared helpers

Normalisation, trigram extraction, Jaccard similarity, and a timing helper. These are the same
functions the shipped implementation uses — `algorithms/inverted_index.py` in the repository.


In [ ]:
def norm(s: str) -> str:
    """Lowercase, expand '&', strip punctuation, collapse whitespace."""
    s = str(s).lower().replace("&", " and ")
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def trigrams(s: str) -> frozenset[str]:
    """Character trigrams, with sentinel padding so short strings still score."""
    p = f"  {s}  "
    return frozenset(p[i:i + 3] for i in range(len(p) - 2))


def jaccard(a: frozenset[str], b: frozenset[str]) -> float:
    """|A ∩ B| / |A ∪ B| over trigram sets."""
    if not a and not b:
        return 0.0
    inter = len(a & b)
    return inter / (len(a) + len(b) - inter)


def timed(fn, repeats: int = REPEATS):
    """Best-of-N wall clock in milliseconds, plus the function's return value."""
    best = float("inf")
    result = None
    for _ in range(repeats):
        t0 = time.perf_counter()
        result = fn()
        best = min(best, time.perf_counter() - t0)
    return best * 1000.0, result


NAME_NORM = [norm(n) for n in NAMES]
SKU_NORM = [norm(s) for s in SKUS]
CAT_TRI = [trigrams(n) for n in NAME_NORM]

print("helpers ready")
print(f"catalogue trigram sets built for {len(CAT_TRI)} items")

## 2 · The query set

The workbook's `LiveClaims` sheet stores the **resolved SKU**, not the raw comment, so a replay of
real typed text is not possible from the data. The query set is therefore built by perturbing real
catalogue names the way buyers actually type them:

| Perturbation | Weight | Example |
|---|---|---|
| 1 typo (delete, swap, substitute, duplicate a character) | 46% | `cinnamoroll` → `cinamoroll` |
| 2 typos | 18% | |
| Dropped word | 18% | `my melody pouch purple` → `my melody purple` |
| Pasted chat noise | 18% | `po claim`, `mine`, `sakin` |

Each query carries its **true answer** — the row index it was derived from — so top-1 accuracy is
measurable rather than asserted.


In [ ]:
def perturb(name: str, rng: random.Random):
    """A typed variant of a catalogue name, plus the kind of error introduced."""
    s = norm(name)
    kind = rng.choices(["one", "two", "drop", "noise"], weights=[46, 18, 18, 18])[0]

    if kind == "noise":
        noise = rng.choice(["po", "claim", "ako", "mine", "sakin"])
        joined = f"{noise} {s}" if rng.random() < 0.5 else f"{s} {noise}"
        return norm(joined), "chat noise"

    if kind == "drop":
        words = s.split()
        if len(words) > 2:
            del words[rng.randrange(len(words))]
            return " ".join(words), "dropped word"
        kind = "one"

    edits = 1 if kind == "one" else 2
    chars = list(s)
    for _ in range(edits):
        if not chars:
            break
        op = rng.choice(["del", "swap", "sub", "dup"])
        i = rng.randrange(len(chars))
        if op == "del" and len(chars) > 4:
            del chars[i]
        elif op == "swap" and i + 1 < len(chars):
            chars[i], chars[i + 1] = chars[i + 1], chars[i]
        elif op == "sub":
            chars[i] = rng.choice("abcdefghijklmnopqrstuvwxyz")
        elif op == "dup":
            chars.insert(i, chars[i])
    out = "".join(chars)
    return (out if out != s else s + "x"), ("1 typo" if kind == "one" else "2 typos")


rng = random.Random(SEED)
QUERIES, KINDS = [], []
for i, n in enumerate(NAMES):
    q, kind = perturb(n, rng)
    if q != NAME_NORM[i]:
        QUERIES.append((q, i))
        KINDS.append(kind)

print(f"queries generated : {len(QUERIES)}")
print()
print("sample of the query set:")
for (q, truth), kind in list(zip(QUERIES, KINDS))[:8]:
    print(f"  {kind:<12} {q!r:<40} -> {SKUS[truth]:<10} {NAMES[truth]}")

## 3 · Benchmark 1 — Prefix trie for POS-style SKU lookup

The owner types the first letters of a product code while a live session is running. A **prefix
tree** answers that in one traversal: descend the characters of the prefix, and the node you land
on already carries every row beneath it.

The baseline is what a developer writes first — `[k for k in keys if k.startswith(p)]`, which
inspects every key.

**Complexity.** Build is O(total key length). A query is O(len(prefix) + output), against the
scan's O(n · len(prefix)).


In [ ]:
class TrieNode:
    __slots__ = ("children", "hits")

    def __init__(self):
        self.children = {}
        self.hits = []


class Trie:
    """Prefix tree keyed on a normalised string, holding row indices."""

    def __init__(self, keys):
        self.root = TrieNode()
        for idx, key in enumerate(keys):
            node = self.root
            node.hits.append(idx)
            for ch in key:
                if ch not in node.children:
                    node.children[ch] = TrieNode()
                node = node.children[ch]
                node.hits.append(idx)

    def prefix(self, prefix):
        node = self.root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return []
        return node.hits


KEYS = [f"{s} {n}" for s, n in zip(SKU_NORM, NAME_NORM)]
build_ms, trie = timed(lambda: Trie(KEYS), repeats=1)

PREFIXES = [s[:max(1, len(s) // 2)] for s in SKU_NORM[:300]]

trie_ms, trie_out = timed(lambda: [tuple(trie.prefix(p)) for p in PREFIXES])
scan_ms, scan_out = timed(
    lambda: [tuple(i for i, k in enumerate(KEYS) if k.startswith(p)) for p in PREFIXES])

# THE ASSERTION: identical answers, or stop.
assert trie_out == scan_out, "trie and linear scan disagree - result is invalid"

avg_hits = sum(len(r) for r in trie_out) / len(PREFIXES)

print(f"keys in trie          : {len(KEYS)}")
print(f"build time            : {build_ms:.2f} ms")
print(f"queries               : {len(PREFIXES)}")
print(f"avg results per query : {avg_hits:.1f}")
print()
print(f"  trie          {trie_ms:8.3f} ms total   {trie_ms / len(PREFIXES):.4f} ms/query")
print(f"  linear scan   {scan_ms:8.1f} ms total   {scan_ms / len(PREFIXES):.4f} ms/query")
print()
print(f"  SPEEDUP       {scan_ms / trie_ms:.0f}x")
print()
print("  identical result sets asserted on every query ✓")

## 4 · Benchmark 2 — Inverted trigram index + Jaccard

This is the engine that maps a messy live-chat claim to a catalogue item. Two stages:

1. **Index construction** (once, at startup): map each trigram to the list of items containing it.
2. **Query**: union the posting lists of the query's trigrams into a candidate pool, then score
   only those candidates with Jaccard similarity.

The baseline is a **full scan** — Jaccard against all 767 items for every query.

Two correctness rules, both learned the hard way:

- The pool is **unioned, not intersected**. A query sharing only one trigram with the right item
  would be excluded by an intersection, and short queries have few trigrams.
- The pool is iterated in **sorted order**. A Python set iterates in hash order, so on an exact
  score tie the index could return a different item than the full scan. Sorting makes the two
  agree by construction, which is what lets the assertion below hold.


In [ ]:
def build_index():
    idx = defaultdict(list)
    for i, ts in enumerate(CAT_TRI):
        for g in ts:
            idx[g].append(i)
    return idx


index_build_ms, index = timed(build_index, repeats=1)


def full_scan():
    out = []
    for q, _ in QUERIES:
        qt = trigrams(q)
        best, best_s = -1, -1.0
        for i, ct in enumerate(CAT_TRI):
            s = jaccard(qt, ct)
            if s > best_s:
                best_s, best = s, i
        out.append(best)
    return out


def indexed():
    out = []
    for q, _ in QUERIES:
        qt = trigrams(q)
        pool = set()
        for g in qt:
            pool.update(index.get(g, ()))
        if not pool:
            pool = set(range(len(CAT_TRI)))
        best, best_s = -1, -1.0
        for i in sorted(pool):
            s = jaccard(qt, CAT_TRI[i])
            if s > best_s:
                best_s, best = s, i
        out.append(best)
    return out


scan_ms, scan_out = timed(full_scan, repeats=3)
idx_ms, idx_out = timed(indexed, repeats=3)

# THE ASSERTION: the index must not change a single answer.
assert scan_out == idx_out, "index changed an answer - the pruning is not valid"

correct = sum(1 for got, (_, truth) in zip(idx_out, QUERIES) if got == truth)

# How much of the catalogue survives pruning - this is what explains the speedup.
pool_sizes = []
for q, _ in QUERIES:
    pool = set()
    for g in trigrams(q):
        pool.update(index.get(g, ()))
    pool_sizes.append(len(pool) if pool else len(CAT_TRI))

print(f"distinct trigrams in index : {len(index)}")
print(f"index build time           : {index_build_ms:.2f} ms (once, at startup)")
print(f"queries                    : {len(QUERIES)}")
print()
print(f"  full scan     {scan_ms:8.1f} ms total   {scan_ms / len(QUERIES):.4f} ms/query")
print(f"  indexed       {idx_ms:8.1f} ms total   {idx_ms / len(QUERIES):.4f} ms/query")
print()
print(f"  SPEEDUP       {scan_ms / idx_ms:.2f}x")
print(f"  TOP-1 ACCURACY {100 * correct / len(QUERIES):.1f}%")
print()
print(f"  candidates surviving pruning : {statistics.mean(pool_sizes):.1f} of {len(CAT_TRI)} "
      f"({100 * (1 - statistics.mean(pool_sizes) / len(CAT_TRI)):.1f}% pruned)")
print()
print("  identical answers asserted on all queries ✓")

### 4.1 · Where the misses are

Top-1 accuracy overall hides which error type is hardest. This is the table that tells the team
what to fix next.


In [ ]:
def match(q):
    qt = trigrams(q)
    pool = set()
    for g in qt:
        pool.update(index.get(g, ()))
    if not pool:
        pool = set(range(len(CAT_TRI)))
    best, best_s = -1, -1.0
    for i in sorted(pool):
        s = jaccard(qt, CAT_TRI[i])
        if s > best_s:
            best_s, best = s, i
    return best


per_kind = defaultdict(lambda: [0, 0])
for (q, truth), kind in zip(QUERIES, KINDS):
    b = per_kind[kind]
    b[1] += 1
    if match(q) == truth:
        b[0] += 1

rows = []
for kind, (hit, n) in sorted(per_kind.items(), key=lambda kv: -kv[1][1]):
    rows.append({"error type": kind, "queries": n,
                 "top-1 correct": hit, "top-1 accuracy %": round(100 * hit / n, 1)})

pd.DataFrame(rows)

## 5 · Benchmark 3 — Finite state machine for stock transitions

Every inventory item is in exactly one status, and only certain moves are legal:

```
AVAILABLE --claim--> RESERVED --pay--> KEPT --pack--> PACKED --dispatch--> SHIPPED
                         |                                              |
                     --cancel--> CANCELLED --restock--> AVAILABLE   --return--> RETURNED
```

Seven states, nine legal transitions, **everything else refused**. The transition table is the
single source of truth.

Three implementations are compared:

| Implementation | What it represents |
|---|---|
| **Dict lookup** | The table, as shipped |
| **if/elif chain** | What a developer drifts into as features accumulate |
| **Linear scan** | "Just loop over the legal list" |

All three must agree on 200,000 random `(state, event)` pairs.


In [ ]:
STATES = ["AVAILABLE", "RESERVED", "KEPT", "PACKED", "SHIPPED", "CANCELLED", "RETURNED"]

LEGAL = {
    ("AVAILABLE", "claim"): "RESERVED",
    ("RESERVED", "pay"): "KEPT",
    ("RESERVED", "cancel"): "CANCELLED",
    ("KEPT", "pack"): "PACKED",
    ("PACKED", "dispatch"): "SHIPPED",
    ("SHIPPED", "receive"): "SHIPPED",
    ("CANCELLED", "restock"): "AVAILABLE",
    ("SHIPPED", "return"): "RETURNED",
    ("RETURNED", "restock"): "AVAILABLE",
}


def chain_transition(state, event):
    """The baseline: one branch per known transition."""
    if state == "AVAILABLE":
        if event == "claim":
            return "RESERVED"
    elif state == "RESERVED":
        if event == "pay":
            return "KEPT"
        elif event == "cancel":
            return "CANCELLED"
    elif state == "KEPT":
        if event == "pack":
            return "PACKED"
    elif state == "PACKED":
        if event == "dispatch":
            return "SHIPPED"
    elif state == "SHIPPED":
        if event == "receive":
            return "SHIPPED"
        elif event == "return":
            return "RETURNED"
    elif state == "CANCELLED":
        if event == "restock":
            return "AVAILABLE"
    elif state == "RETURNED":
        if event == "restock":
            return "AVAILABLE"
    return None


TRIALS = 200_000
events_pool = sorted({e for _, e in LEGAL} | {"bogus"})
rng2 = random.Random(SEED)
cases = [(rng2.choice(STATES), rng2.choice(events_pool)) for _ in range(TRIALS)]


def via_table():
    return [LEGAL.get((s, e)) for s, e in cases]


def via_chain():
    return [chain_transition(s, e) for s, e in cases]


def via_scan():
    legal_list = list(LEGAL.items())
    out = []
    for s, e in cases:
        got = None
        for (fs, fe), to in legal_list:
            if fs == s and fe == e:
                got = to
                break
        out.append(got)
    return out


tbl_ms, tbl_out = timed(via_table)
chn_ms, chn_out = timed(via_chain)
scn_ms, scn_out = timed(via_scan)

# THE ASSERTION: all three agree on every case.
assert tbl_out == chn_out, "table and chain disagree"
assert tbl_out == scn_out, "table and scan disagree"

legal_n = sum(1 for v in tbl_out if v is not None)

print(f"states                  : {len(STATES)}")
print(f"legal transitions       : {len(LEGAL)}")
print(f"random cases tested     : {TRIALS:,}")
print(f"  legal outcomes        : {legal_n:,}")
print(f"  rejected outcomes     : {TRIALS - legal_n:,}")
print()
print(f"  dict lookup   {tbl_ms:7.2f} ms   {tbl_ms * 1000 / TRIALS:.3f} us/op")
print(f"  if/elif chain {chn_ms:7.2f} ms   {chn_ms * 1000 / TRIALS:.3f} us/op")
print(f"  linear scan   {scn_ms:7.2f} ms   {scn_ms * 1000 / TRIALS:.3f} us/op")
print()
print(f"  SPEEDUP vs chain : {chn_ms / tbl_ms:.2f}x")
print(f"  SPEEDUP vs scan  : {scn_ms / tbl_ms:.2f}x")
print()
print("  all three implementations agree on every case ✓")

## 6 · Benchmark 4 — Min-heap order deadline expiry

The client's invoice prints a 24-hour payment rule. Every unpaid claim carries a deadline; when one
lapses, the order cancels itself and its items return to stock. The app must find lapsed deadlines
continuously while a session is running.

- **Min-heap:** ordered by deadline. Each tick pops whatever has expired — O(k log n) where *k* is
  the number expiring, not the number pending.
- **Baseline:** rebuild the pending list every tick — O(n) per tick.

10,000 orders over 1,440 one-minute ticks (24 hours).


In [ ]:
N_ORDERS = 10_000
MINUTES = 1_440

rng3 = random.Random(7)
deadlines = {f"ORD-{i:04d}": rng3.uniform(0, MINUTES) for i in range(N_ORDERS)}


def via_heap():
    heap = [(d, oid) for oid, d in deadlines.items()]
    heapq.heapify(heap)
    due = []
    for minute in range(MINUTES):
        while heap and heap[0][0] <= minute:
            _, oid = heapq.heappop(heap)
            due.append(oid)
    return due


def via_scan():
    pending = list(deadlines.items())
    due = []
    for minute in range(MINUTES):
        keep = []
        for oid, d in pending:
            if d <= minute:
                due.append(oid)
            else:
                keep.append((oid, d))
        pending = keep
    return due


heap_ms, heap_due = timed(via_heap, repeats=1)
scan_ms, scan_due = timed(via_scan, repeats=1)

# THE ASSERTION: both cancel the same orders.
assert sorted(heap_due) == sorted(scan_due), "heap and scan disagree"

print(f"orders                  : {N_ORDERS:,}")
print(f"ticks (minutes)         : {MINUTES:,}")
print(f"orders that lapsed      : {len(heap_due):,}")
print()
print(f"  min-heap      {heap_ms:8.1f} ms   {heap_ms / MINUTES:.3f} ms/tick")
print(f"  linear rescan {scan_ms:8.1f} ms   {scan_ms / MINUTES:.3f} ms/tick")
print()
print(f"  SPEEDUP       {scan_ms / heap_ms:.1f}x")
print()
print("  identical cancellation sets asserted ✓")

## 7 · Scaling — does the index earn its keep as the catalogue grows?

The asymptotic claim has to be measured, not asserted. Both matchers run at increasing catalogue
sizes on the same query set.

This is the **most honest slide in the deck**: the inverted index buys only ~1.7× because pruning
removes roughly half the catalogue while Jaccard scoring still dominates the remaining cost. The
ratio is flat across a 4× size change, which is the finding — not a disappointment.


In [ ]:
rows = []
for n in (191, 383, 767):
    if n > len(NAME_NORM):
        continue
    sub_norm = NAME_NORM[:n]
    sub_tri = CAT_TRI[:n]
    sub_index = defaultdict(list)
    for i, ts in enumerate(sub_tri):
        for g in ts:
            sub_index[g].append(i)

    sub_q = [(q, t) for q, t in QUERIES if t < n]
    if not sub_q:
        continue

    def full():
        out = []
        for q, _ in sub_q:
            qt = trigrams(q)
            best, best_s = -1, -1.0
            for i, ct in enumerate(sub_tri):
                s = jaccard(qt, ct)
                if s > best_s:
                    best_s, best = s, i
            out.append(best)
        return out

    def idx():
        out = []
        for q, _ in sub_q:
            qt = trigrams(q)
            pool = set()
            for g in qt:
                pool.update(sub_index.get(g, ()))
            if not pool:
                pool = set(range(n))
            best, best_s = -1, -1.0
            for i in sorted(pool):
                s = jaccard(qt, sub_tri[i])
                if s > best_s:
                    best_s, best = s, i
            out.append(best)
        return out

    f_ms, f_out = timed(full, repeats=3)
    i_ms, i_out = timed(idx, repeats=3)
    assert f_out == i_out, f"scaling run disagrees at n={n}"

    rows.append({"catalogue size": n, "queries": len(sub_q),
                 "full scan ms/query": round(f_ms / len(sub_q), 4),
                 "indexed ms/query": round(i_ms / len(sub_q), 4),
                 "speedup": round(f_ms / i_ms, 2)})

pd.DataFrame(rows)

## 8 · Summary against the report

The right-hand column is what the ALGOCOM PBL Milestone 1 report quotes from the run on the group's
own hardware. Absolute times differ between machines; the ratios and accuracy are stable.


In [ ]:
REPORT = {
    "Trie prefix lookup": ("121x", "0.0004 vs 0.0442 ms/query"),
    "Inverted index + Jaccard": ("1.7x, 97.0% top-1", "55.4% of catalogue pruned"),
    "FSM transition validation": ("1.40x vs chain, 3.53x vs scan", "0.109 us/op"),
    "Min-heap deadline expiry": ("63.5x", "6.4 ms vs 409.0 ms"),
}

summary = [
    {"algorithm": "Trie prefix lookup",
     "live result": f"{scan_ms / trie_ms:.0f}x",
     "correctness": "identical result sets",
     "report quotes": REPORT["Trie prefix lookup"][0]},
    {"algorithm": "Inverted index + Jaccard",
     "live result": f"{scan_ms / idx_ms:.2f}x, {100 * correct / len(QUERIES):.1f}% top-1",
     "correctness": "identical answers",
     "report quotes": REPORT["Inverted index + Jaccard"][0]},
    {"algorithm": "FSM transition validation",
     "live result": f"{chn_ms / tbl_ms:.2f}x vs chain",
     "correctness": "all three agree",
     "report quotes": REPORT["FSM transition validation"][0]},
    {"algorithm": "Min-heap deadline expiry",
     "live result": f"{scan_ms / heap_ms:.1f}x",
     "correctness": "identical cancellation sets",
     "report quotes": REPORT["Min-heap deadline expiry"][0]},
]

pd.DataFrame(summary)

## 9 · Conclusion

Four algorithms, four measured speedups, and **four correctness assertions that all passed**. The
assertions are the point: each implementation was proven to return exactly what its naive baseline
returns, so the speedup is a speedup on the same answer rather than a faster way to be wrong.

The two findings worth defending in the oral exam:

1. **Trigram Jaccard beats edit distance** on this domain (97.0% against 74.0%) because the errors
   are word-level — dropped words and pasted chat noise — which an edit-distance metric cannot see.
2. **The inverted index buys only 1.7×**, not the order-of-magnitude a textbook implies, because
   pruning removes 55% of candidates while Jaccard scoring dominates the rest. Reporting that
   honestly is stronger than quoting the theoretical figure.

**Repository:** https://github.com/WhiteQilin/momompurin-live-manager

The same implementations ship as importable modules under `algorithms/`, with a runnable demo
(`python demo/run_demo.py`) that carries these assertions outside the notebook.
